In [42]:
import pandas as pd
import numpy as np
import json
import time
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
class CFG:
    # данные
    DATA_PATH = "/Users/test/Desktop/GP5/DL_project_5/data/nlp_dataset.csv" # путь к CSV 
    TEXT_COL  = "clean_text"  
    TARGET    = "Issue"   

    # DagsHub / MLflow
    DAGSHUB_OWNER = "kulikanton05"
    DAGSHUB_REPO  = "GP5"

    # train / val / test
    TEST_SIZE    = 0.2  
    VAL_SIZE     = 0.1   
    RANDOM_STATE = 42 

    # словарь и последовательности для RNN
    MAX_LEN   = 300
    MIN_FREQ  = 5
    MAX_VOCAB = 20000

    # обучение RNN
    EMBED_DIM  = 200 
    HIDDEN_DIM = 128 
    NUM_LAYERS = 1
    DROPOUT    = 0.3
    RNN_EPOCHS = 8
    RNN_BATCH  = 64
    RNN_LR     = 1e-3
    CLIP_NORM  = 1.0 

    # обучение DistilBERT
    BERT_MODEL  = "distilbert-base-uncased"
    BERT_MAXLEN = 128
    BERT_EPOCHS = 2
    BERT_BATCH  = 8
    BERT_LR     = 2e-5
    
def seed_everything(seed=CFG.RANDOM_STATE):
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

In [3]:
df = pd.read_csv(CFG.DATA_PATH)
df

,Consumer complaint narrative,Product,Issue,word_count,clean_text
0,I mobile deposited a XXXX check to Ameris Bank...,Checking or savings account,Managing an account,188,i mobile deposited a check to ameris bank on y...
1,"Credence Resource Management , LLC ( Use the a...",Debt collection,Attempts to collect debt not owed,332,credence resource management llc use the addre...
2,"Help my rights are being violated, 15 U.S.C. 1...",Debt collection,Took or threatened to take negative or legal a...,85,help my rights are being violated u s c g b of...
3,This company is violating my rights they have ...,Debt collection,Took or threatened to take negative or legal a...,74,this company is violating my rights they have ...
4,My information was heisted and this accounts a...,Credit card,Other,31,my information was heisted and this accounts a...
...,...,...,...,...,...
53893,My credit reports are inaccurate. These inaccu...,Credit reporting or other personal consumer re...,Incorrect information on your report,39,my credit reports are inaccurate these inaccur...
53894,I received a very shady voicemail from structu...,Debt collection,Other,39,i received a very shady voicemail from structu...
53895,In XXXX of XXXX I was charged for a Best Buy r...,Credit card,Problem with a purchase shown on your statement,434,in of i was charged for a best buy renewal of ...
53896,"My name is XXXX XXXX, residing at XXXX XXXX XX...",Debt collection,Attempts to collect debt not owed,277,my name is residing at ga my date of birth is ...


In [26]:
df["text_for_issue"] = (df["Product"].astype(str) + " " + df["clean_text"].astype(str))
df

,Consumer complaint narrative,Product,Issue,word_count,clean_text,text_for_issue
0,I mobile deposited a XXXX check to Ameris Bank...,Checking or savings account,Managing an account,188,i mobile deposited a check to ameris bank on y...,Checking or savings account i mobile deposited...
1,"Credence Resource Management , LLC ( Use the a...",Debt collection,Attempts to collect debt not owed,332,credence resource management llc use the addre...,Debt collection credence resource management l...
2,"Help my rights are being violated, 15 U.S.C. 1...",Debt collection,Took or threatened to take negative or legal a...,85,help my rights are being violated u s c g b of...,Debt collection help my rights are being viola...
3,This company is violating my rights they have ...,Debt collection,Took or threatened to take negative or legal a...,74,this company is violating my rights they have ...,Debt collection this company is violating my r...
4,My information was heisted and this accounts a...,Credit card,Other,31,my information was heisted and this accounts a...,Credit card my information was heisted and thi...
...,...,...,...,...,...,...
53893,My credit reports are inaccurate. These inaccu...,Credit reporting or other personal consumer re...,Incorrect information on your report,39,my credit reports are inaccurate these inaccur...,Credit reporting or other personal consumer re...
53894,I received a very shady voicemail from structu...,Debt collection,Other,39,i received a very shady voicemail from structu...,Debt collection i received a very shady voicem...
53895,In XXXX of XXXX I was charged for a Best Buy r...,Credit card,Problem with a purchase shown on your statement,434,in of i was charged for a best buy renewal of ...,Credit card in of i was charged for a best buy...
53896,"My name is XXXX XXXX, residing at XXXX XXXX XX...",Debt collection,Attempts to collect debt not owed,277,my name is residing at ga my date of birth is ...,Debt collection my name is residing at ga my d...


In [67]:
df = df[df[CFG.TARGET] != 'Other']
df[CFG.TARGET].value_counts()

Issue
Attempts to collect debt not owed                               12106
Managing an account                                              6592
Written notification about debt                                  3973
Problem with a purchase shown on your statement                  3915
False statements or representation                               3119
Took or threatened to take negative or legal action              2049
Incorrect information on your report                             1911
Problem with a lender or other company charging your account     1748
Closing an account                                               1597
Fraud or scam                                                    1526
Name: count, dtype: int64

In [68]:
df.shape

(38536, 6)

In [69]:
le = LabelEncoder()
y = le.fit_transform(df[CFG.TARGET])
X = df[CFG.TEXT_COL]

In [70]:
dict(zip(le.classes_, le.transform(le.classes_)))

{'Attempts to collect debt not owed': np.int64(0),
 'Closing an account': np.int64(1),
 'False statements or representation': np.int64(2),
 'Fraud or scam': np.int64(3),
 'Incorrect information on your report': np.int64(4),
 'Managing an account': np.int64(5),
 'Problem with a lender or other company charging your account': np.int64(6),
 'Problem with a purchase shown on your statement': np.int64(7),
 'Took or threatened to take negative or legal action': np.int64(8),
 'Written notification about debt': np.int64(9)}

In [71]:
class_names = list(le.classes_)
num_classes = len(class_names)
class_names, num_classes

(['Attempts to collect debt not owed',
  'Closing an account',
  'False statements or representation',
  'Fraud or scam',
  'Incorrect information on your report',
  'Managing an account',
  'Problem with a lender or other company charging your account',
  'Problem with a purchase shown on your statement',
  'Took or threatened to take negative or legal action',
  'Written notification about debt'],
 10)

In [72]:
X_trv, X_test, y_trv, y_test = train_test_split(X, y, test_size=CFG.TEST_SIZE, random_state=CFG.RANDOM_STATE, stratify=y)
val_rel = CFG.VAL_SIZE/ (1.0 - CFG.TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_trv, y_trv, test_size=val_rel, random_state=CFG.RANDOM_STATE, stratify=y_trv)

In [73]:
len(X_train), len(X_val), len(X_test)

(26974, 3854, 7708)

In [52]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

mps


In [74]:
def save_split_csv(X, y, part_name):
    # Функция сохраняет один сплит в CSV: текст, закодированный и изначальный таргет
    out = pd.DataFrame({
        CFG.TEXT_COL: X,
        "label": y,
        CFG.TARGET: le.inverse_transform(y)
    })
    path = f'/Users/test/Desktop/GP5/DL_project_5/artifacts_issue/{part_name}.csv'
    out.to_csv(path, index=False)
    return path

SPLIT_PATHS = {
    "train": save_split_csv(X_train, y_train, "train"),
    "val":   save_split_csv(X_val,   y_val,   "val"),
    "test":  save_split_csv(X_test,  y_test,  "test"),
}
print("Сохранены сплиты:", SPLIT_PATHS)

Сохранены сплиты: {'train': '/Users/test/Desktop/GP5/DL_project_5/artifacts_issue/train.csv', 'val': '/Users/test/Desktop/GP5/DL_project_5/artifacts_issue/val.csv', 'test': '/Users/test/Desktop/GP5/DL_project_5/artifacts_issue/test.csv'}


In [ ]:
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"
PAD_IDX,   UNK_IDX   = 0, 1


def tokenize(text):
    return str(text).split()


def build_vocab(texts, min_freq=CFG.MIN_FREQ, max_vocab=CFG.MAX_VOCAB):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    kept = [w for w, c in counter.most_common() if c >= min_freq]
    kept = kept[: max_vocab - 2]

    stoi = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for word in kept:
        stoi[word] = len(stoi)
    return stoi


def encode(text, stoi, max_len=CFG.MAX_LEN):
    ids = [stoi.get(tok, UNK_IDX) for tok in tokenize(text)][:max_len]
    if not ids:
        ids = [UNK_IDX]
    return ids

stoi = build_vocab(X_train)
vocab_size = len(stoi)
print("Размер словаря:", vocab_size)

Размер словаря: 9537


In [76]:
list(stoi.items())[:30]

[('<pad>', 0),
 ('<unk>', 1),
 ('the', 2),
 ('i', 3),
 ('to', 4),
 ('and', 5),
 ('of', 6),
 ('a', 7),
 ('my', 8),
 ('that', 9),
 ('this', 10),
 ('account', 11),
 ('was', 12),
 ('not', 13),
 ('is', 14),
 ('on', 15),
 ('in', 16),
 ('credit', 17),
 ('debt', 18),
 ('for', 19),
 ('or', 20),
 ('with', 21),
 ('they', 22),
 ('have', 23),
 ('me', 24),
 ('as', 25),
 ('reporting', 26),
 ('am', 27),
 ('from', 28),
 ('it', 29)]

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len=CFG.MAX_LEN):
        self.encoded = [encode(t, stoi, max_len) for t in texts]
        self.labels  = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], int(self.labels[idx])

In [ ]:
def collate_batch(batch):
    seqs, labels = zip(*batch)

    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.int64)
    max_len = int(lengths.max())
    padded = torch.full((len(seqs), max_len), PAD_IDX, dtype=torch.int64)
    for i, s in enumerate(seqs):
        padded[i, : len(s)] = torch.tensor(s, dtype=torch.int64)

    labels = torch.tensor(labels, dtype=torch.int64)
    return padded, lengths, labels

train_ds = TextDataset(X_train, y_train, stoi)
val_ds   = TextDataset(X_val,   y_val,   stoi)
test_ds  = TextDataset(X_test,  y_test,  stoi)

train_loader = DataLoader(train_ds, batch_size=CFG.RNN_BATCH, shuffle=True,  collate_fn=collate_batch)
val_loader   = DataLoader(val_ds,   batch_size=CFG.RNN_BATCH, shuffle=False, collate_fn=collate_batch)
test_loader  = DataLoader(test_ds,  batch_size=CFG.RNN_BATCH, shuffle=False, collate_fn=collate_batch)


In [79]:
xb, lb, yb = next(iter(train_loader))
print("padded:", xb.shape, "lengths:", lb.shape, "labels:", yb.shape)

padded: torch.Size([64, 300]) lengths: torch.Size([64]) labels: torch.Size([64])


In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":        accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":    recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro":        f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    preds, trues = [], []

    for text, lengths, labels in loader:
        text, labels = text.to(device), labels.to(device) 

        optimizer.zero_grad()
        logits = model(text, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_NORM)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.detach().cpu().tolist())

    metrics = compute_metrics(trues, preds)
    metrics["loss"] = total_loss / max(n, 1)
    return metrics


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    preds, trues = [], []

    for text, lengths, labels in loader:
        text, labels = text.to(device), labels.to(device)
        logits = model(text, lengths)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.detach().cpu().tolist())

    metrics = compute_metrics(trues, preds)
    metrics["loss"] = total_loss / max(n, 1)
    return metrics


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    preds, trues = [], []
    for text, lengths, labels in loader:
        text = text.to(device)
        logits = model(text, lengths)
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.tolist())
    return compute_metrics(trues, preds), trues, preds


In [17]:
import dagshub
import mlflow

dagshub.init(repo_owner=CFG.DAGSHUB_OWNER, repo_name=CFG.DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment("Issue_Classification")
print("Tracking URI:", mlflow.get_tracking_uri())

/Users/test/Desktop/GP5/DL_project_5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as kulikanton05

Initialized MLflow to track repo "kulikanton05/GP5"

Repository kulikanton05/GP5 initialized!

Tracking URI: https://dagshub.com/kulikanton05/GP5.mlflow


In [18]:
def log_rnn_run(model, model_name, history, test_metrics, y_true, y_pred):
    '''Записывает один полный RNN-эксперимент в MLflow + DagsHub.'''
    with mlflow.start_run(run_name=model_name):
        # Параметры эксперимента
        mlflow.log_params({
            "model_name":   model_name,
            "target":       CFG.TARGET,
            "batch_size":   CFG.RNN_BATCH,
            "epochs":       CFG.RNN_EPOCHS,
            "learning_rate": CFG.RNN_LR,
            "embedding_dim": CFG.EMBED_DIM,
            "hidden_dim":   CFG.HIDDEN_DIM,
            "vocab_size":   vocab_size,
            "max_len":      CFG.MAX_LEN,
        })

        # Метрики по эпохам
        for ep, h in enumerate(history, start=1):
            mlflow.log_metrics({
                "train_loss":      h["train_loss"],
                "val_loss":        h["val_loss"],
                "val_accuracy":    h["val_accuracy"],
                "val_f1_macro":    h["val_f1_macro"],
            }, step=ep)

        # Итоговые метрики на тесте
        mlflow.log_metrics({
            "accuracy":        test_metrics["accuracy"],
            "precision_macro": test_metrics["precision_macro"],
            "recall_macro":    test_metrics["recall_macro"],
            "f1_macro":        test_metrics["f1_macro"],
        })

        # classification report как текстовый артефакт
        report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
        rep_path = f"/Users/test/Desktop/GP5/DL_project_5/artifacts_issue/{model_name}_report.txt"
        with open(rep_path, "w") as f:
            f.write(report)
        mlflow.log_artifact(rep_path, artifact_path="reports")

        # Артефакты для воспроизводимости: модель, LabelEncoder, словарь
        mlflow.pytorch.log_model(model, artifact_path="model")

        le_path = "/Users/test/Desktop/GP5/DL_project_5/artifacts_issue/label_classes.json"
        with open(le_path, "w") as f:
            json.dump(class_names, f, ensure_ascii=False)
        mlflow.log_artifact(le_path, artifact_path="assets")

        vocab_path = "/Users/test/Desktop/GP5/DL_project_5/artifacts_issue/vocab.json"
        with open(vocab_path, "w") as f:
            json.dump(stoi, f)
        mlflow.log_artifact(vocab_path, artifact_path="assets")

        # Данные эксперимента
        for p in SPLIT_PATHS.values():
            mlflow.log_artifact(p, artifact_path="data")

    print(f"[{model_name}] залогирован в MLflow.")


In [19]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes,
                 embed_dim=CFG.EMBED_DIM, hidden_dim=CFG.HIDDEN_DIM,
                 num_layers=CFG.NUM_LAYERS, dropout=CFG.DROPOUT, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, bidirectional=False, batch_first=True,)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text, lengths):
        embedded = self.embedding(text)         
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        last_hidden = self.dropout(hidden[-1])              
        return self.fc(last_hidden)    

In [20]:
seed_everything()
lstm_model = LSTMClassifier(vocab_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=CFG.RNN_LR)

lstm_history = []
for epoch in range(1, CFG.RNN_EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch(lstm_model, train_loader, criterion, optimizer)
    va = validate_epoch(lstm_model, val_loader, criterion)
    lstm_history.append({
        "train_loss": tr["loss"], "val_loss": va["loss"],
        "val_accuracy": va["accuracy"], "val_f1_macro": va["f1_macro"],
    })
    print(f"[LSTM] epoch {epoch}/{CFG.RNN_EPOCHS} | {time.time()-t0:4.1f}s "
          f"| train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} "
          f"| val_acc {va['accuracy']:.3f} | val_f1 {va['f1_macro']:.3f}")

[LSTM] epoch 1/8 | 269.0s | train_loss 1.6149 | val_loss 1.4002 | val_acc 0.492 | val_f1 0.271
[LSTM] epoch 2/8 | 253.0s | train_loss 1.2876 | val_loss 1.2067 | val_acc 0.574 | val_f1 0.375
[LSTM] epoch 3/8 | 259.5s | train_loss 1.1214 | val_loss 1.1193 | val_acc 0.601 | val_f1 0.390
[LSTM] epoch 4/8 | 254.7s | train_loss 1.0054 | val_loss 1.0781 | val_acc 0.616 | val_f1 0.439
[LSTM] epoch 5/8 | 263.3s | train_loss 0.9118 | val_loss 1.0854 | val_acc 0.629 | val_f1 0.470
[LSTM] epoch 6/8 | 264.5s | train_loss 0.8232 | val_loss 1.0969 | val_acc 0.624 | val_f1 0.482
[LSTM] epoch 7/8 | 255.1s | train_loss 0.7418 | val_loss 1.1190 | val_acc 0.625 | val_f1 0.499
[LSTM] epoch 8/8 | 254.4s | train_loss 0.6621 | val_loss 1.1396 | val_acc 0.627 | val_f1 0.512


In [21]:
lstm_test, lstm_true, lstm_pred = evaluate_model(lstm_model, test_loader)
print("LSTM test metrics:", {k: round(v, 4) for k, v in lstm_test.items()})
print()
print(classification_report(lstm_true, lstm_pred, target_names=class_names, zero_division=0))

LSTM test metrics: {'accuracy': 0.634, 'precision_macro': 0.5671, 'recall_macro': 0.5005, 'f1_macro': 0.5243}

                                                              precision    recall  f1-score   support

                           Attempts to collect debt not owed       0.69      0.80      0.74      2421
                                          Closing an account       0.63      0.35      0.45       319
                          False statements or representation       0.44      0.37      0.40       624
                                               Fraud or scam       0.44      0.33      0.38       305
                        Incorrect information on your report       0.63      0.53      0.58       382
                                         Managing an account       0.57      0.61      0.59      1318
                                                       Other       0.67      0.74      0.70      3073
Problem with a lender or other company charging your account       0.32 

In [22]:
log_rnn_run(lstm_model, "LSTM", lstm_history, lstm_test, lstm_true, lstm_pred)

2026/06/15 19:26:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 19:26:59 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run LSTM at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/1/runs/cab0d7f1befb4384ae33bbb6bb88206d
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/1
[LSTM] залогирован в MLflow.


In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=128, num_filters=128, filter_sizes=(2, 3, 4, 5), dropout=0.5, pad_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(0.2)
        self.convs = nn.ModuleList([nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs) for fs in filter_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, text, lengths=None):
        embedded = self.embedding(text)
        embedded = embedded.permute(0, 2, 1)

        conv_outputs = []
        for conv in self.convs:
            x = torch.relu(conv(embedded))
            x = torch.max(x, dim=2).values
            conv_outputs.append(x)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)

        return self.fc(x)

In [82]:
seed_everything()

cnn_model = TextCNN(
    vocab_size=vocab_size,
    num_classes=num_classes,
    embed_dim=200,
    num_filters=256,
    filter_sizes=(2, 3, 4, 5),
    dropout=0.3,
    pad_idx=PAD_IDX
).to(device)

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=CFG.RNN_LR, weight_decay=1e-4)

cnn_history = []

for epoch in range(1, CFG.RNN_EPOCHS + 1):
    t0 = time.time()

    tr = train_epoch(cnn_model, train_loader, criterion, optimizer)
    va = validate_epoch(cnn_model, val_loader, criterion)

    cnn_history.append({
        "train_loss": tr["loss"],
        "val_loss": va["loss"],
        "val_accuracy": va["accuracy"],
        "val_f1_macro": va["f1_macro"],
    })

    print(
        f"[TextCNN] epoch {epoch}/{CFG.RNN_EPOCHS} | {time.time()-t0:4.1f}s "
        f"| train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} "
        f"| val_acc {va['accuracy']:.3f} | val_f1 {va['f1_macro']:.3f}"
    )

[TextCNN] epoch 1/8 | 72.6s | train_loss 1.3740 | val_loss 1.0048 | val_acc 0.580 | val_f1 0.557
[TextCNN] epoch 2/8 | 70.8s | train_loss 1.0495 | val_loss 1.0395 | val_acc 0.593 | val_f1 0.554
[TextCNN] epoch 3/8 | 69.9s | train_loss 0.9191 | val_loss 0.9799 | val_acc 0.591 | val_f1 0.579
[TextCNN] epoch 4/8 | 69.7s | train_loss 0.8255 | val_loss 0.9217 | val_acc 0.641 | val_f1 0.620
[TextCNN] epoch 5/8 | 69.8s | train_loss 0.7278 | val_loss 0.9264 | val_acc 0.606 | val_f1 0.588
[TextCNN] epoch 6/8 | 69.9s | train_loss 0.6372 | val_loss 0.9110 | val_acc 0.660 | val_f1 0.609
[TextCNN] epoch 7/8 | 69.5s | train_loss 0.5417 | val_loss 0.8890 | val_acc 0.665 | val_f1 0.633
[TextCNN] epoch 8/8 | 70.0s | train_loss 0.4676 | val_loss 0.9104 | val_acc 0.641 | val_f1 0.620


In [83]:
cnn_test, cnn_true, cnn_pred = evaluate_model(cnn_model, test_loader)

print("TextCNN test metrics:", {k: round(v, 4) for k, v in cnn_test.items()})
print()

print(classification_report(
    cnn_true,
    cnn_pred,
    target_names=class_names,
    zero_division=0
))

TextCNN test metrics: {'accuracy': 0.6432, 'precision_macro': 0.6019, 'recall_macro': 0.6703, 'f1_macro': 0.617}

                                                              precision    recall  f1-score   support

                           Attempts to collect debt not owed       0.95      0.56      0.70      2421
                                          Closing an account       0.54      0.74      0.63       319
                          False statements or representation       0.36      0.72      0.48       624
                                               Fraud or scam       0.51      0.70      0.59       305
                        Incorrect information on your report       0.56      0.84      0.67       382
                                         Managing an account       0.72      0.63      0.67      1319
Problem with a lender or other company charging your account       0.38      0.40      0.39       350
             Problem with a purchase shown on your statement       0.

In [84]:
log_rnn_run(cnn_model, "TextCNN_3", cnn_history, cnn_test, cnn_true, cnn_pred)

2026/06/16 02:35:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 02:35:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run TextCNN_3 at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/1/runs/c072328b68764061b0a36e2c00af9206
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/1
[TextCNN_3] залогирован в MLflow.
